# Session-4: Full CDF + DFDC training

Detector at scale. Extracts ALL Celeb-DF clips (~5,000), trains the multimodal
detector with ImageNet-pretrained ResNet50 backbone, target val AUC 0.85+.

Total runtime: ~3.5 h
* Bootstrap + restore: 3 min
* Extract ~5,000 CDF clips with GPU MTCNN: ~50 min
* Train 20 epochs on ~4,500 train + ~500 val: ~2.5 h
* Save + push: 1 min

The result lands as `best_session4.pt` in `dfdc-smoke-artifacts` v3.

## 1. Bootstrap (clone, install, restore data, CDF symlinks)

In [ ]:
import os, shutil, subprocess

os.chdir("/kaggle/working")
if not os.path.exists("code"):
    subprocess.run(["git", "clone", "https://github.com/Parthivsurya/deepfake-detection.git", "code"], check=True)
os.chdir("/kaggle/working/code")
subprocess.run(["git", "pull", "origin", "main"])

subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"])
subprocess.run(["pip", "install", "-q", "--no-deps", "facenet-pytorch"])

# Restore from preserved dataset (DFDC frames + manifests + smoke checkpoint)
SRC = "/kaggle/input/datasets/parthivsuryakb/dfdc-smoke-artifacts"
assert os.path.exists(SRC), f"attach dfdc-smoke-artifacts first - {SRC} missing"
for name in ["frames", "audio", "manifests"]:
    src_dir = f"{SRC}/{name}_dfdc_smoke"
    if os.path.lexists(name):
        if os.path.islink(name) or os.path.isfile(name): os.remove(name)
        else: shutil.rmtree(name)
    shutil.copytree(src_dir, name)
os.makedirs("checkpoints", exist_ok=True)

# CDF symlinks (the dataset has a doubled-folder quirk from how it was zipped)
CDF_SRC = "/kaggle/input/datasets/parthivsuryakb/celeb-df-v2"
assert os.path.exists(CDF_SRC), f"attach celeb-df-v2 first - {CDF_SRC} missing"
os.makedirs("/kaggle/working/cdf_root", exist_ok=True)
for sub in ["Celeb-real", "Celeb-synthesis"]:
    nested = f"{CDF_SRC}/{sub}/{sub}"
    link = f"/kaggle/working/cdf_root/{sub}"
    if not os.path.lexists(link):
        os.symlink(nested, link)

print("=== ready ===")
print("pwd:", os.getcwd())
print("DFDC frames already restored:",
      len([p for p in os.listdir("frames") if os.path.isdir(f"frames/{p}")]))
print("cdf_root contents:", os.listdir("/kaggle/working/cdf_root"))
subprocess.run(["git", "log", "--oneline", "-3"])


## 2. Verify GPU

In [ ]:
import torch
print("CUDA :", torch.cuda.is_available())
print("Name :", torch.cuda.get_device_name(0))
print("Count:", torch.cuda.device_count())


## 3. Build combined DFDC + CDF manifests (full, no subset)

In [ ]:
%cd /kaggle/working/code
!python scripts/prepare_datasets.py \
    --dataset dfdc celebdf \
    --root /kaggle/input/competitions/deepfake-detection-challenge/train_sample_videos \
           /kaggle/working/cdf_root \
    --out manifests/

import pandas as pd
for split in ["train", "val", "test"]:
    df = pd.read_csv(f"manifests/{split}.csv")
    print(f"\n{split}: total {len(df)}")
    print(f"  by dataset: {df['dataset'].value_counts().to_dict()}")
    print(f"  by label:   {df['label'].value_counts().to_dict()}")


## 4. Identify which clips need extraction (DFDC partial done, all CDF new)

In [ ]:
%cd /kaggle/working/code
import pandas as pd
from pathlib import Path

have = {p.name for p in Path("frames").glob("*") if p.is_dir()}
print(f"already extracted: {len(have)} clips")

for split in ["train", "val", "test"]:
    df = pd.read_csv(f"manifests/{split}.csv")
    missing = df[~df["clip_id"].isin(have)]
    missing.to_csv(f"manifests/{split}.s4.todo.csv", index=False)
    print(f"{split}: {len(missing)}/{len(df)} clips to extract")

total_todo = sum(len(pd.read_csv(f'manifests/{s}.s4.todo.csv')) for s in ['train', 'val', 'test'])
print(f"\nTotal to extract: {total_todo} clips (~{total_todo * 0.6 / 60:.1f} min on GPU)")


## 5. Extract all missing clips (slow step, ~50 min for ~5000 clips with GPU MTCNN)

In [ ]:
%cd /kaggle/working/code
import subprocess
for split in ["train", "val", "test"]:
    print(f"\n=== extracting {split} ===")
    ret = subprocess.run([
        "python", "scripts/extract_frames.py",
        "--manifest", f"manifests/{split}.s4.todo.csv",
        "--out_frames", "frames/", "--out_audio", "audio/",
        "--fps", "4", "--max_frames", "32", "--crop_size", "224",
    ])
    print(f"{split} exit: {ret.returncode}")


## 6. Build combined extracted manifests (full data, no subset)

In [ ]:
%cd /kaggle/working/code
import pandas as pd
from pathlib import Path

for split in ["train", "val", "test"]:
    parts = []
    for p in [f"manifests/{split}.small.extracted.csv",       # session-1 DFDC
              f"manifests/{split}.s4.todo.extracted.csv"]:    # session-4 new extractions
        if Path(p).exists():
            parts.append(pd.read_csv(p))
    full = pd.concat(parts, ignore_index=True).drop_duplicates(subset=["clip_id"])

    # Sanity: drop rows whose frames_dir doesn't actually exist
    full["_have"] = full["clip_id"].apply(lambda c: Path(f"frames/{c}").exists())
    n_drop = (~full["_have"]).sum()
    if n_drop:
        print(f"  WARN dropping {n_drop} {split} rows with missing frames")
    full = full[full["_have"]].drop(columns=["_have"])

    full.to_csv(f"manifests/{split}.s4.extracted.csv", index=False)
    print(f"{split}: {len(full)} clips ready")
    print(f"  by dataset: {full['dataset'].value_counts().to_dict()}")
    print(f"  by label:   {full['label'].value_counts().to_dict()}")


## 7. Train detector with ResNet50 backbone (20 epochs, batch 8, balanced sampler)

In [ ]:
%cd /kaggle/working/code
import yaml, os

cfg = yaml.safe_load(open("configs/default.yaml"))  # backbone: resnet50, balanced_sampling: True
cfg["data"]["manifest_train"] = "manifests/train.s4.extracted.csv"
cfg["data"]["manifest_val"]   = "manifests/val.s4.extracted.csv"
cfg["data"]["batch_size"]     = 8
cfg["data"]["num_workers"]    = 2
cfg["train"]["epochs"]        = 20
cfg["train"]["lr"]            = 1.0e-4
cfg["train"]["weight_decay"]  = 0.05
yaml.safe_dump(cfg, open("configs/session4.yaml", "w"))

# Train fresh (delete any prior best.pt so it tracks only this run)
if os.path.exists("checkpoints/best.pt"):
    os.remove("checkpoints/best.pt")

!python scripts/train.py --config configs/session4.yaml --device cuda


## 8. Save checkpoint as notebook output + push as v3 of dfdc-smoke-artifacts

In [ ]:
%cd /kaggle/working/code
import shutil, json, os
shutil.copy("checkpoints/best.pt", "/kaggle/working/best_session4.pt")
!ls -lh /kaggle/working/best_session4.pt

# Stage upload directory and push as new version of preserved Kaggle Dataset
os.makedirs("/kaggle/working/upload4", exist_ok=True)
shutil.copy("/kaggle/working/best_session4.pt", "/kaggle/working/upload4/best_session4.pt")
with open("/kaggle/working/upload4/dataset-metadata.json", "w") as fp:
    json.dump({"title": "dfdc-smoke-artifacts",
               "id": "parthivsuryakb/dfdc-smoke-artifacts",
               "licenses": [{"name": "CC0-1.0"}]}, fp)
!kaggle datasets version -p /kaggle/working/upload4 -m "v3: full CDF + DFDC with ResNet50 backbone"


## 9. Quick val confidence check (paste output if you want me to sanity-check)

In [ ]:
%cd /kaggle/working/code
import torch, yaml
from scripts.train import build_model
from data.datasets import VideoManifest, VideoClipDataset

cfg = yaml.safe_load(open("configs/session4.yaml"))
model = build_model(cfg).to("cuda")
ckpt = torch.load("checkpoints/best.pt", map_location="cuda", weights_only=False)
model.load_state_dict(ckpt["model"], strict=False)
model.eval()

val_m = VideoManifest.load("manifests/val.s4.extracted.csv")
val_ds = VideoClipDataset(val_m,
    num_frames=cfg["data"]["num_frames"], frame_size=cfg["data"]["frame_size"],
    audio_sample_rate=cfg["data"]["audio_sample_rate"], audio_seconds=cfg["data"]["audio_seconds"])

for label_name, label_val in [("REAL", 0), ("FAKE", 1)]:
    print(f"\n=== 5 {label_name} clips ===")
    idx = val_m.df[val_m.df["label"] == label_val].index[:5].tolist()
    for i in idx:
        item = val_ds[i]
        with torch.no_grad():
            out = model(item["frames"].unsqueeze(0).to("cuda"),
                        item["audio"].unsqueeze(0).to("cuda"),
                        has_audio=item["has_audio"].unsqueeze(0).to("cuda"))
        p = torch.softmax(out["logits"], dim=-1)[0].cpu()
        print(f"  {item['clip_id'][:40]:40s}  P(fake)={p[1]:.3f}")

print(f"\n[best.pt val metrics at save] {ckpt.get('metrics')}")
